JAMfit DIII-D VDE Example, Part 2: Reconstructing and Post-processing the VDE {#doc_tCurr_jamfit_vde_reconstruct}
==========
This example reconstructs the synthetic DIII-D vertical displacement event (VDE) made in part 1 (\ref doc_tCurr_jamfit_vde_create). Run part 1 first: this notebook reads the files it writes. Once part 1 is done, it runs in under a minute.

**What this shows**
1. Loading the saved reduced wall model (`initialize_reduced_model`) and the data generated from part 1. 
2. Reconstructing the filament currents and the wall mode amplitudes at every time with the three JAMfit solvers: least squares with Tikhonov regularization (`run_reconstruction_lstsq`), truncated SVD with a Laplacian (`run_reconstruction_tsvd_laplace`) and a direct Laplacian (`run_reconstruction_laplace`).
3. Getting the wall psi on the TokaMaker grid from the flux probes added in part 1.
4. Post-processing using the Laplace reconstruction with TokaMaker (`post_process_tidx`): total psi, LCFS, limiting point, q95, internal inductance li and current centroid at each time.

**The setup**
- **Data** (from part 1): the signals of the 117 real DIII-D magnetics, the 24 F and E coil currents and the total plasma current, at 151 times over 30 ms. The synthetic data has no noise.
- **Model:** the reduced wall model with 4 wall modes, and the 101 filaments read back from the coil XML.
- **TokaMaker mesh:** `DIIID_tokamaker_JAMfit_model.h5`, the same mesh as in part 1, used for the limiter and the post-processing.

## Load ThinCurr library

To load the ThinCurr python module we need to tell python where to the module is located. This can be done either through the `PYTHONPATH` environment variable or within a script using `sys.path.append()` as below, where we look for the environement variable `OFT_ROOTPATH` to provide the path to where the OpenFUSIONToolkit is installed (`/Applications/OFT` for binaries on macOS).

In [ ]:
import os
import sys
import re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import xml.etree.ElementTree as ET
from scipy.interpolate import griddata
plt.rcParams['figure.figsize'] = (6, 4)
%matplotlib inline

# Use the OFT install pointed to by OFT_ROOTPATH, if set
oft_python_path = os.getenv('OFT_ROOTPATH')
if oft_python_path is not None:
    sys.path.append(os.path.join(oft_python_path, 'python'))
from OpenFUSIONToolkit import OFT_env
from OpenFUSIONToolkit.ThinCurr.JAMfit import JAMfit, get_bbox_grid_pts
from OpenFUSIONToolkit.TokaMaker.meshing import load_gs_mesh

myOFT = OFT_env(nthreads=4)

## User inputs
These must match part 1: the same meshes, the coil XML with the filaments, the reduced model, and the same flux probe grid (`probe_n_r`, `probe_n_z`). The solver weights are set in each solver's section below.

In [ ]:
meshfile_thincurr = 'DIIID_meshfile_thincurr.h5'          # ThinCurr mesh of the DIII-D vessel
meshfile_tokamaker = 'DIIID_tokamaker_JAMfit_model.h5'    # 2D TokaMaker mesh (limiter, coils)
xml_name = 'DIIID_JAMfit_wfil_ex.xml'                     # coil XML with the plasma filaments, from part 1
reduced_filename = 'DIIID_vde_reduced_model_ex.h5'        # reduced wall model, from part 1
B0 = 1.91503608   # vacuum toroidal field at R0 [T]
R0 = 1.69550002   # major radius [m]
probe_n_r = 15    # flux probe grid points along R used in part 1
probe_n_z = 15    # flux probe grid points along Z used in part 1

## Loading the model and the data
The JAMfit object here does not need `setup_JAMfit`: the reconstruction only uses the reduced model, which \ref OpenFUSIONToolkit.ThinCurr.JAMfit.JAMfit.initialize_reduced_model "initialize_reduced_model()" loads from file. Creating the JAMfit object still sets up its TokaMaker (`jam_obj.mygs`), which gives the limiter and does the post-processing.

The saved data, all from part 1:

| File | Content | Shape |
|---|---|---|
| `vde_diiid_sensor_measurements.npy` | signals of the 117 real sensors, in the order of the sensor file (Mirnov `[T]`, flux loops `[Wb]`) | (151, 117) |
| `vde_diiid_coil_currs.npy` | F and E coil currents `[A]` | (151, 24) |
| `vde_diiid_totalip.npy` | total plasma current `[A]` | (151,) |
| `vde_diiid_time_array_sensors.npy` | times of the sensor signals `[s]` | (151,) |
| `vde_diiid_time_array_coilcurr.npy` | times of the coil currents `[s]` | (151,) |

Here the total plasma current is known exactly; in an experiment it comes from a Rogowski coil.

In [ ]:
jam_obj = JAMfit(xml_name, meshfile_thincurr, meshfile_tokamaker, B0=B0, R0=R0, oft_env=myOFT)
jam_obj.initialize_reduced_model(reduced_filename)
limiter = jam_obj.mygs.lim_contour   # (n_pts, 2) array of (R, Z) [m]

magnetic_data = np.load('vde_diiid_sensor_measurements.npy')   # shape (n_times, num_sensors)
time_array_sensors = np.load('vde_diiid_time_array_sensors.npy')
time_array_coils = np.load('vde_diiid_time_array_coilcurr.npy')
totalip = np.load('vde_diiid_totalip.npy')                     # [A], shape (n_times,)
coilcurrs = np.load('vde_diiid_coil_currs.npy')                # [A], shape (n_times, num_non_fil_coils)

## Quantities used by all solvers
- `rgrid`, `zgrid`: the filament positions `[m]`, read back from the unnamed coil sets in the XML by the helper `get_grid_from_xml` below (it is not part of JAMfit).
- `num_non_fil_coils`: the number of F and E coils. Their currents are known, so their flux is subtracted from the signals.
- `num_sensors`: the number of real sensors. The reduced model also contains the flux probes after these, which are not measurements, so the solvers only use the first `num_sensors`.
- `sigma`: the expected error of each sensor, in the same units as its signal. Every sensor row is divided by its sigma, so each sensor enters the fit in units of its own error and no one sensor dominates.

Here sigma is 5% of each sensor's mean absolute signal over time, but at least 2% of the RMS over all sensors and times:

$\sigma_i = \max\left(0.05 \, \overline{\lvert s_i \rvert}, \; 0.02 \, s_{rms}\right)$

The synthetic data has no noise, so here sigma only sets the relative weight of the sensors.

In [ ]:
def get_grid_from_xml(xml_path):
    '''! Read the filament positions back from a JAMfit coil XML.
    The filaments are the coil sets without a name (the shaping coils are named).

    @param xml_path str, path to the coil XML
    @result numpy.ndarray, R of the filaments `[m]`, shape (n_fil,)
    @result numpy.ndarray, Z of the filaments `[m]`, shape (n_fil,)
    '''
    root = ET.parse(xml_path).getroot()
    r_grid = []
    z_grid = []
    for coil_set in root.iter('coil_set'):
        if coil_set.get('name') is None:
            for coil in coil_set.findall('coil'):
                r, z = [float(v.strip()) for v in coil.text.split(',')]
                r_grid.append(r)
                z_grid.append(z)
    return np.array(r_grid), np.array(z_grid)

In [ ]:
rgrid, zgrid = get_grid_from_xml(xml_name)
num_filaments = len(rgrid)
num_non_fil_coils = coilcurrs.shape[1]
num_sensors = magnetic_data.shape[1]
rms_global = np.sqrt(np.mean(magnetic_data**2))
sigma = np.maximum(0.05 * np.abs(magnetic_data).mean(axis=0), 0.02 * rms_global)
print('{0} filaments, {1} shaping coils, {2} sensors, {3} times'.format(num_filaments, num_non_fil_coils, num_sensors, len(time_array_sensors)))

## Plotting helper
The same two plots are made for each solver: the reconstructed Ip against the measured Ip, and the filament currents at a few times. It also prints the mean Ip error, the time average of $\lvert I_p - I_{p,rec} \rvert / I_p$.

In [ ]:
def plot_solution(sol_fil, title, num_frames=6):
    '''! Plot the reconstructed Ip over time and the filament currents at num_frames times.

    @param sol_fil numpy.ndarray, filament currents `[A]`, shape (n_times, n_fil)
    @param title str, solver name for the plot titles
    @param num_frames int, number of times to show the filament currents at (default: 6)
    '''
    ip_rec = np.sum(sol_fil, axis=1)
    fig, ax = plt.subplots()
    ax.plot(time_array_sensors * 1E3, totalip / 1E3, label='measured Ip')
    ax.plot(time_array_sensors * 1E3, ip_rec / 1E3, 'k--', label='reconstructed Ip')
    ax.set_xlabel('time [ms]')
    ax.set_ylabel('Ip [kA]')
    ax.set_ylim(0, 1.1 * max(np.max(totalip), np.max(ip_rec)) / 1E3)
    ax.set_title(title)
    ax.legend()
    plt.show()
    print('{0}: mean Ip error {1:.2f} %'.format(title, 100 * np.mean(np.abs(totalip - ip_rec) / totalip)))

    frames = np.linspace(0, len(time_array_sensors) - 1, num_frames).astype(int)
    fig, axes = plt.subplots(1, num_frames, figsize=(2.6 * num_frames, 4.5))
    for ax, i in zip(axes, frames):
        ax.plot(limiter[:, 0], limiter[:, 1], 'k-')
        sc = ax.scatter(rgrid, zgrid, c=sol_fil[i] / 1E3, cmap='plasma', s=25, edgecolors='k', linewidths=0.3)
        ax.set_aspect('equal', adjustable='box')
        ax.set_title('t = {0:.1f} ms'.format(time_array_sensors[i] * 1E3), fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
        fig.colorbar(sc, ax=ax, orientation='horizontal', pad=0.03, label='[kA]')
    fig.suptitle(title + ': filament currents')
    plt.tight_layout()
    plt.show()

## Solver 1: least squares with Tikhonov regularization
\ref OpenFUSIONToolkit.ThinCurr.JAMfit.JAMfit.run_reconstruction_lstsq "run_reconstruction_lstsq()" solves for the wall mode amplitudes $I_w$ and the filament currents $I_f$ together, one time at a time. It sets up one row per sensor, plus one row for the total plasma current:

$\begin{bmatrix} M_s^T & M_{sc,f}^T \\ 0 & w_{Ip} \mathbf{1}^T \end{bmatrix} \begin{bmatrix} I_w \\ I_f \end{bmatrix} = \begin{bmatrix} s - M_{sc,c}^T I_c \\ w_{Ip} I_p \end{bmatrix}$

| Symbol | Meaning |
|---|---|
| $s$ | measured sensor signals |
| $M_s$ | signal on each sensor per unit amplitude of each wall mode (`Ms` of the reduced model) |
| $M_{sc,f}$, $M_{sc,c}$ | signal on each sensor per amp in each filament, and in each F and E coil (`Msc`) |
| $I_c$ | F and E coil currents `[A]` (known) |
| $I_p$ | measured total plasma current `[A]` |
| $w_{Ip}$ | Ip row weight, `ip_weight` divided by $\max(\lvert I_p \rvert, 1\,\mathrm{kA})$ |

Each sensor row is divided by that sensor's sigma. Writing this as $A x = B$, the solver minimises

$\lVert A x - B \rVert^2 + \alpha_w^2 \lVert I_w \rVert^2 + \alpha_f^2 \lVert I_f \rVert^2$

where $\alpha_w$ = `reg_factor_wall` and $\alpha_f$ = `reg_factor_fil` are the Tikhonov regularization weights. Because the Ip row is divided by $I_p$, an Ip error of 1/`ip_weight` (here 0.25%) costs the same as missing one sensor by one sigma.

**Note:** The weights here were found by hand. With real data, sweep them and compare the residual with the noise level to pick them.

In [ ]:
ip_weight = 4E2         # weight of the Ip constraint row
reg_factor_fil = 4E-4   # Tikhonov weight on the filament currents
reg_factor_wall = 1E-6  # Tikhonov weight on the wall mode amplitudes

solution_lstsq_fil_tidx = []
solution_lstsq_wall_tidx = []
residual_lstsq_tidx = []
for t_idx in range(len(time_array_sensors)):
    solution_fil, solution_wall, residual, Ax, B = jam_obj.run_reconstruction_lstsq(
        magnetic_data[t_idx, :], totalip[t_idx], num_non_fil_coils, coilcurrs[t_idx, :], ip_weight, sigma,
        reg_factor_fil, reg_factor_wall, num_sensors)
    solution_lstsq_fil_tidx.append(solution_fil)
    solution_lstsq_wall_tidx.append(solution_wall)
    residual_lstsq_tidx.append(residual)
solution_lstsq_fil_tidx = np.array(solution_lstsq_fil_tidx)
solution_lstsq_wall_tidx = np.array(solution_lstsq_wall_tidx)
residual_lstsq_tidx = np.array(residual_lstsq_tidx)

In [ ]:
plot_solution(solution_lstsq_fil_tidx, 'least squares')

The least-squares solution matches Ip to about 0.1 %, but the current peaks on the outboard filaments, nearest the sensors, rather than in the middle of the plasma, and about 15 % of the filament currents are negative. There are 105 unknowns (101 filaments and 4 wall modes) against 117 sensors, and many filament patterns give nearly the same signals, so the fit is poorly conditioned. Tikhonov regularization only makes the currents smaller, not smoother. (With both regularization weights above zero the system can always be solved, even with more unknowns than sensors.)

## Solver 2: truncated SVD with a Laplacian
Instead of solving for every filament current, the TSVD solver writes them as a sum of the strongest singular vectors of the weighted filament-to-sensor matrix $M_{sc,f}/\sigma$:

$I_f = U_k c$

where the columns of $U_k$ are the first $k$ = `num_modes` left singular vectors (each one a current pattern over the filaments) and $c$ holds their $k$ unknown weights. Each singular vector is a current pattern the sensors see well; the weak ones, which the sensors barely see, are dropped. These are TSVD modes, not the wall modes of part 1.

The sensor rows and the Ip row are as in solver 1, with $U_k c$ in place of $I_f$. The solver then minimises

$\lVert A x - B \rVert^2 + \beta_w^2 \lVert I_w \rVert^2 + \beta_L^2 \lVert L U_k c \rVert^2$

where $L$ is the finite-difference Laplacian on the filament grid, $\beta_L$ = `lap_lam` and $\beta_w$ = `reg_wall`. The Laplacian term penalises sharp changes between neighbouring filaments, so it smooths the current. At the edge of the grid it acts as if the current just outside were zero, which pushes current away from the edge.

First, \ref OpenFUSIONToolkit.ThinCurr.JAMfit.JAMfit.prepare_tsvd_laplace "prepare_tsvd_laplace()" builds the matrices once; they do not change with time. With `verbose=True` it plots the singular values. A sharp drop in the singular values marks a good `num_modes`. Here there is no sharp drop, a sign that TSVD is not an ideal fit for this problem.

In [ ]:
num_modes = 38   # TSVD modes kept, well below the 117 sensors

Msc_coils_weighted, Ms_weighted, U_trun, ls_mat, ls_mat_fil, lap_proj, ip_row, N = jam_obj.prepare_tsvd_laplace(
    sigma, num_non_fil_coils, num_sensors, rgrid, zgrid, num_modes, verbose=True)

Now we run \ref OpenFUSIONToolkit.ThinCurr.JAMfit.JAMfit.run_reconstruction_tsvd_laplace "run_reconstruction_tsvd_laplace()" at every time. Here the Ip row weight `lam` is set by hand; with `lam=None` it is set automatically, as in solver 3. The weights were found by hand, and changing `num_modes` changes the best `lam`, `lap_lam` and `reg_wall`.

In [ ]:
lam = 1E-3       # weight of the Ip constraint row
lap_lam = 1E-6   # Laplacian smoothing weight on the filaments
reg_wall = 1E-3  # Tikhonov weight on the wall mode amplitudes

solution_tsvd_fil_tidx = []
solution_tsvd_wall_tidx = []
for t_idx in range(len(time_array_sensors)):
    curr_fil, curr_wall, Ax, diagnostics = jam_obj.run_reconstruction_tsvd_laplace(
        magnetic_data[t_idx, :], totalip[t_idx], coilcurrs[t_idx, :], Msc_coils_weighted, Ms_weighted, U_trun,
        ls_mat, ls_mat_fil, lap_proj, ip_row, N, sigma, lam, lap_lam, reg_wall)
    solution_tsvd_fil_tidx.append(curr_fil)
    solution_tsvd_wall_tidx.append(curr_wall)
solution_tsvd_fil_tidx = np.array(solution_tsvd_fil_tidx)
solution_tsvd_wall_tidx = np.array(solution_tsvd_wall_tidx)

In [ ]:
plot_solution(solution_tsvd_fil_tidx, 'TSVD + Laplacian')

TSVD gives a similar picture to least squares: the current still peaks on the outboard edge, so the profile is hollow, and more of the filament currents are negative (about 22 %). The solver knows nothing about plasma physics, so it may favour current patterns that sit close to the outer magnetics.

## Solver 3: direct Laplacian
\ref OpenFUSIONToolkit.ThinCurr.JAMfit.JAMfit.run_reconstruction_laplace "run_reconstruction_laplace()" solves for every filament current directly (no SVD step). It minimises the same expression as solver 2, with $I_f$ in place of $U_k c$:

$\lVert A x - B \rVert^2 + \beta_w^2 \lVert I_w \rVert^2 + \beta_L^2 \lVert L I_f \rVert^2$

With `lam=None` the Ip row weight is set automatically from the size of the signals,

$\lambda = 100 \cdot 10^{\lfloor \log_{10} \left( \overline{\lvert B \rvert} / \max(\lvert I_p \rvert, 1\,\mathrm{kA}) \right) \rfloor}$

where $\overline{\lvert B \rvert}$ is the mean of the weighted sensor signals with the coil flux removed. This puts the Ip row on the same order as the sensor rows, and the factor 100 gives the single Ip row more weight against the many sensor rows.

In [ ]:
lap_lam_direct = 1E-6   # Laplacian smoothing weight on the filaments
reg_wall_direct = 1E-5  # Tikhonov weight on the wall mode amplitudes

solution_lap_fil_tidx = []
solution_lap_wall_tidx = []
for t_idx in range(len(time_array_sensors)):
    curr_fil, curr_wall, Ax, diagnostics = jam_obj.run_reconstruction_laplace(
        magnetic_data[t_idx, :], totalip[t_idx], coilcurrs[t_idx, :], sigma, num_non_fil_coils, num_sensors,
        rgrid, zgrid, lam=None, lap_lam=lap_lam_direct, reg_wall=reg_wall_direct)
    solution_lap_fil_tidx.append(curr_fil)
    solution_lap_wall_tidx.append(curr_wall)
solution_lap_fil_tidx = np.array(solution_lap_fil_tidx)
solution_lap_wall_tidx = np.array(solution_lap_wall_tidx)

In [ ]:
plot_solution(solution_lap_fil_tidx, 'direct Laplacian')

## Post-processing with TokaMaker
\ref OpenFUSIONToolkit.ThinCurr.JAMfit.JAMfit.post_process_tidx "post_process_tidx()" turns one time slice of a reconstruction into equilibrium quantities, using TokaMaker. Its inputs, for one time slice:

| Input | Type | Description |
|---|---|---|
| `filaments_at_time` | numpy.ndarray | filament currents `[A]`, shape (n_fil,) |
| `wall_psi` | numpy.ndarray | psi from the wall currents on every TokaMaker node `[Wb/rad]` (see below) |
| `coil_curr_dict` | dict | F and E coil currents `[A]`, with keys that match the TokaMaker coil names **exactly** |
| `rgrid`, `zgrid` | numpy.ndarray | filament positions `[m]` |
| `disk_radius` | float | radius of the current disk TokaMaker puts at each filament `[m]` (default 0.3) |
| `psi_basis` | numpy.ndarray | psi of 1 A in each filament, to skip placing the disks every time (optional, see below) |
| `touch_tol` | float | largest LCFS-limiter distance that counts as touching `[m]` (default 0.0005) |

It returns the LCFS, the point where the LCFS touches the limiter (`None` if it does not), psi at the LCFS, the total psi on the TokaMaker grid, q, q95, li, the current centroid, the geometric centroid of the LCFS and the area inside it.

The plasma psi is built from current disks at the positive filaments only: TokaMaker cannot represent a negative plasma current. The current centroid uses all filaments, and li uses the current enclosed by the LCFS.

We post-process the TSVD solution here; any of the three solutions can be used.

**F and E coil currents by TokaMaker coil name**
In the TokaMaker mesh, the E coils are split into numbered pieces (for example `ECOILA` is stored as 48 regions, `ECOILA1` to `ECOILA48`); the F coils are not split. The helper `get_diiid_dict_at_tidx` below removes the trailing number from each region name and gives that coil its current at one time.

In [ ]:
coil_names = ['F1A', 'F2A', 'F3A', 'F4A', 'F5A', 'F6A', 'F7A', 'F8A', 'F9A',
              'F1B', 'F2B', 'F3B', 'F4B', 'F5B', 'F6B', 'F7B', 'F8B', 'F9B',
              'ECOILA', 'ECOILB', 'E567UP', 'E567DN', 'E89DN', 'E89UP']   # same order as the coil currents
_, _, _, coil_dict, _ = load_gs_mesh(meshfile_tokamaker)

def get_diiid_dict_at_tidx(coil_dict, coilcurrs, t_idx, coil_names):
    '''! Coil current dictionary for post_process_tidx at one time, for the DIII-D TokaMaker mesh.

    @param coil_dict dict, coil regions of the TokaMaker mesh (from load_gs_mesh)
    @param coilcurrs numpy.ndarray, shaping coil currents `[A]`, shape (n_times, n_coils), columns in coil_names order
    @param t_idx int, time index
    @param coil_names list, coil names in the column order of coilcurrs
    @result dict, coil name (trailing digits removed) -> current `[A]`
    '''
    name_to_idx = {name: i for i, name in enumerate(coil_names)}
    coil_curr_dict = {}
    for coil in coil_dict.keys():
        base_name = re.sub(r'\d+$', '', coil)   # strip the trailing piece number
        coil_curr_dict[base_name] = coilcurrs[t_idx, name_to_idx[base_name]]
    return coil_curr_dict

**Wall psi from the flux probes**
JAMfit does not solve for the wall currents directly but for the amplitudes of the wall modes. To turn these into psi, part 1 put a nonphysical toroidal flux loop (a probe) at each point of a uniform `probe_n_r` x `probe_n_z` grid over the TokaMaker mesh, after the real sensors. The reduced model's `Ms` couples every sensor, probes included, to the wall modes, so the wall psi at the probes is

$\psi_w = M_{s,p}^T I_w / 2\pi$

where $M_{s,p}$ holds the probe columns of `Ms` (the columns after `num_sensors`) and the $2\pi$ turns flux `[Wb]` into TokaMaker psi `[Wb/rad]`. This is what \ref OpenFUSIONToolkit.ThinCurr.JAMfit.JAMfit.get_wall_psi_tidx "get_wall_psi_tidx()" returns.

`get_bbox_grid_pts`, with the same grid size as part 1, gives the probe positions. Below, the probe values are interpolated linearly onto every node of `jam_obj.mygs`, the TokaMaker model that `post_process_tidx` adds the wall psi to. The grid spans the whole mesh, so every node is inside it; the nearest probe is used for any node the linear interpolation misses.

In [ ]:
probe_pts, _, _, _ = get_bbox_grid_pts(meshfile_tokamaker, myOFT, n_r=probe_n_r, n_z=probe_n_z)
assert jam_obj.torus_reduced.Ms.shape[1] == num_sensors + len(probe_pts), 'probes do not match part 1'
# copy: mygs.r is a view on TokaMaker memory
node_pts = jam_obj.mygs.r[:, :2].copy()   # TokaMaker nodes the wall psi is needed on, shape (n_nodes, 2) [m]


def wall_psi_on_grid(solution_wall_at_time):
    '''! Wall psi on every TokaMaker node, interpolated linearly from the flux probe grid.

    @param solution_wall_at_time numpy.ndarray, wall mode amplitudes at one time, shape (n_wall,)
    @result numpy.ndarray, wall psi `[Wb/rad]`, shape (n_nodes,)
    '''
    wall_psi_probes = jam_obj.get_wall_psi_tidx(num_sensors, solution_wall_at_time)
    wall_psi = griddata(probe_pts, wall_psi_probes, node_pts, method='linear')
    # a node outside the convex hull of the probes (only possible by round-off at the grid edge) gets NaN; use the nearest probe
    nan_mask = np.isnan(wall_psi)
    if np.any(nan_mask):
        wall_psi[nan_mask] = griddata(probe_pts, wall_psi_probes, node_pts[nan_mask], method='nearest')
    return wall_psi

**Post-processing every time slice**
TokaMaker's psi is linear in the filament currents, so we compute the psi of 1 A in each filament once with \ref OpenFUSIONToolkit.ThinCurr.JAMfit.JAMfit.compute_psi_basis "compute_psi_basis()" and pass it as `psi_basis`. This gives the same result as letting `post_process_tidx` place every filament itself, and is much faster over many time slices.

In [ ]:
psi_basis = jam_obj.compute_psi_basis(rgrid, zgrid)   # default disk_radius, same as post_process_tidx

lcfs_pts_tidx = []
limiting_pts_tidx = []
total_psi_tidx = []
q95_tidx = []
internal_inductance_tidx = []
current_centroid_tidx = []
area_tidx = []
for t_idx in range(len(time_array_sensors)):
    coil_curr_dict = get_diiid_dict_at_tidx(coil_dict, coilcurrs, t_idx, coil_names)
    lcfs_pts, lim_pts, _, total_psi, _, q95, li, curr_cent, _, area = jam_obj.post_process_tidx(
        solution_tsvd_fil_tidx[t_idx], wall_psi_on_grid(solution_tsvd_wall_tidx[t_idx]), coil_curr_dict,
        rgrid, zgrid, psi_basis=psi_basis)
    lcfs_pts_tidx.append(lcfs_pts)
    limiting_pts_tidx.append(lim_pts)
    total_psi_tidx.append(total_psi)
    q95_tidx.append(q95)
    internal_inductance_tidx.append(li)
    current_centroid_tidx.append(curr_cent)
    area_tidx.append(area)

**Total psi at selected times**
The total psi (filaments + coils + wall) on the TokaMaker grid, with the limiting point (red x, only when the LCFS touches the limiter) and the current centroid (green square). The light blue dots are the filaments.

In [ ]:
selected_idx = [0, 8, 40, 60, 80, 100]
triang = mtri.Triangulation(jam_obj.mygs.r[:, 0], jam_obj.mygs.r[:, 1], jam_obj.mygs.lc[:, :3])
fig, axes = plt.subplots(2, 3, figsize=(10, 10))
for ax, j in zip(axes.flatten(), selected_idx):
    ax.scatter(rgrid, zgrid, color='#ADD8E6', s=10, zorder=0, alpha=0.6)
    ax.tricontourf(triang, total_psi_tidx[j], levels=100, cmap='RdBu_r')
    ax.tricontour(triang, total_psi_tidx[j], levels=20, colors='k', linewidths=0.5, alpha=0.4)
    if limiting_pts_tidx[j] is not None:   # (R, Z) where the LCFS touches the limiter, None otherwise
        ax.scatter(limiting_pts_tidx[j][0], limiting_pts_tidx[j][1], label='limiting point', marker='x', color='r', s=60, zorder=3)
    ax.scatter(current_centroid_tidx[j][0], current_centroid_tidx[j][1], label='current centroid', marker='s', color='g', s=60, zorder=4)
    ax.plot(limiter[:, 0], limiter[:, 1], 'k-', zorder=1)
    ax.set_aspect('equal')
    ax.set_title('t = {0:.1f} ms'.format(time_array_sensors[j] * 1E3), fontsize=9)
    ax.legend(loc='lower center', fontsize=6, framealpha=1)
plt.tight_layout()
plt.show()

**Equilibrium quantities over time**
The current centroid moves down through the VDE, from Z = 0 to about Z = -0.94 m (in part 1 the Gaussian centre moves from Z = 0 to Z = -1.1 m). q95 falls as the plasma is pushed into the lower wall. li is noisier than q95. q95 and li are `nan` at any time where no LCFS is found; in this run an LCFS is found at every time.

In [ ]:
current_centroid_tidx = np.array(current_centroid_tidx)
fig, axes = plt.subplots(3, 1, figsize=(6, 7), sharex=True)
axes[0].plot(time_array_sensors * 1E3, current_centroid_tidx[:, 1])
axes[0].set_ylabel('centroid Z [m]')
axes[1].plot(time_array_sensors * 1E3, q95_tidx)
axes[1].set_ylabel('q95')
axes[2].plot(time_array_sensors * 1E3, internal_inductance_tidx)
axes[2].set_ylabel('li')
axes[2].set_xlabel('time [ms]')
for ax in axes:
    ax.grid(True)
plt.tight_layout()
plt.show()